# TopiOCQA Retrieval Index Creator

This notebook prepares the sparse and dense retrieval indexes for TopiOCQA.

### Local checkout setup

Use this public checkout on a local machine or a prepared Colab runtime. The next cell locates it; it does not clone a private repository, mount Drive, or request credentials.

In [ ]:
from pathlib import Path
import sys
import os

# Use an existing public checkout, locally or in Colab.
# No private repository, deploy key, or Drive mount is required.
ROOT = Path(os.environ.get("ROCC_ROOT", os.environ.get("MASTER_THESIS_PROJECT_ROOT", Path.cwd()))).resolve()
if not (ROOT / "experiments" / "rocc").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "experiments" / "rocc").exists())
os.environ["ROCC_ROOT"] = str(ROOT)
os.environ["MASTER_THESIS_PROJECT_ROOT"] = str(ROOT)
sys.path.insert(0, str(ROOT / "experiments"))
print("project:", ROOT)


### Runtime setup

Locates the `experiments` directory, exposes the ROCC API, and initializes the local or Colab runtime.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

EXPERIMENTS_DIR = ROOT / "experiments"
if str(EXPERIMENTS_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_DIR))

import rocc
from rocc.paths import project_path

notebook_runtime = rocc.bootstrap_notebook()
colab_runtime = notebook_runtime.qrecc_runtime
pd.set_option("display.max_colwidth", 180)
display(pd.DataFrame([notebook_runtime.to_dict()]))


### Prepare public TopiOCQA inputs

Reuse local data and complete downloaded indices. Public train/dev data and the corpus are downloaded only when needed by the selected preparation stage.

In [ ]:
TOPIOCQA_DATA_DIR = project_path("experiments", "data", "topiocqa")

topiocqa_setup = rocc.prepare_topiocqa_resources_for_indexing(
    data_dir=TOPIOCQA_DATA_DIR,
    check_azure_ance=False,
    progress=True,
    timeout=1000,
)
topiocqa_resources = topiocqa_setup.resources
topiocqa_artifacts = topiocqa_setup.artifacts
topiocqa_bm25_status_before = topiocqa_setup.bm25_status
topiocqa_azure_ance_status = None

TOPIOCQA_BM25_INDEX_DIR = topiocqa_artifacts.bm25_index_dir
TOPIOCQA_BM25_LOG_FILE = topiocqa_setup.bm25_log_file
TOPIOCQA_ANCE_LOCAL_INDEX_DIR = topiocqa_artifacts.ance_index_dir

display(pd.DataFrame([topiocqa_setup.to_dict()]))
display(pd.DataFrame([topiocqa_artifacts.to_dict()]))


### TopiOCQA dataset

Loads the normalized TopiOCQA frame and reports its dimensions and split distribution. Example rows verify conversational fields, gold passage IDs, and topic-switch metadata.

In [4]:
topiocqa_frame = rocc.load_topiocqa_frame(topiocqa_resources)

print("shape:", topiocqa_frame.shape)
display(topiocqa_frame["split"].value_counts().rename_axis("split").reset_index(name="turns"))
display(topiocqa_frame[[
    "split",
    "conv_id",
    "turn_id",
    "question",
    "answers",
    "positive_ctx_passage_ids",
    "topic_document",
    "topic_switch",
]].head(5))


shape: (47964, 16)


,split,turns
0,train,45450
1,dev,2514


,split,conv_id,turn_id,question,answers,positive_ctx_passage_ids,topic_document,topic_switch
0,dev,1,1,when will the new dunkirk film be released on dvd,[18 December 2017],[14979903],Dunkirk (2017 film),False
1,dev,1,2,what is this film about,[Dunkirk evacuation of World War II],[14979882],Dunkirk (2017 film),False
2,dev,1,3,can you mention a few members of the cast,"[Fionn Whitehead, Tom Glynn-Carney, Jack Lowden, Harry Styles]",[14979882],Dunkirk (2017 film),False
3,dev,1,4,where was it shot,"[Dunkirk and Urk, Netherlands]",[14979895],Dunkirk (2017 film),False
4,dev,1,5,can you tell me anything about the latter place's past,[Until 1475 the High and Low Lordship of Urk and Emmeloord was in the hands of the Van Kuinre family],[25289152],Urk,True


TopiOCQA contains **47,964 turns**, divided into **45,450 training** and **2,514 development** turns. The example conversation remains on *Dunkirk* for four turns before switching to *Urk* at turn 5. The `topic_switch` flag records this transition while gold passage IDs identify the relevant retrieval targets.

### Persisted ANCE encoding analysis

`load_ance_passage_encoding_analysis()` reads the previously generated JSON artifact without loading the corpus or rerunning tokenization. It reconstructs five pandas tables covering corpus size, token-length distribution, unknown tokens, truncation, and batch padding. The original run metadata is restored alongside these summaries.

The next cell displays the saved original corpus-encoding profile, not measurements of a new index build.

In [6]:
ANCE_ANALYSIS_OUT = project_path(
    "experiments",
    "results",
    "ance_passage_encoding",
    "topiocqa.json",
)

ance_passage_analysis = rocc.load_ance_passage_encoding_analysis(ANCE_ANALYSIS_OUT)

display(ance_passage_analysis.corpus_summary)
display(ance_passage_analysis.token_length_summary)
display(ance_passage_analysis.unk_summary)
display(ance_passage_analysis.truncation_summary)
display(ance_passage_analysis.padding_summary)


,dataset_name,num_passages,expected_total_passages,max_passages,full_scan,batch_size,max_length_setting,workers,chunk_batches,encoder_name,source_file
0,topiocqa,25700592,25700592,None,True,160,384,32,4,castorini/ance-msmarco-passage,/experiments/data/topiocqa/downloads/data/wikipedia_split/full_wiki_segments.tsv


,mean,variance,ci95_low,ci95_high,min,max
0,99.487888,2920.525609,99.466994,99.508782,3,12779


,unk_tokens,visible_tokens,unk_rate,passages_with_unk,passages_with_unk_fraction
0,0,2556897619,0.0,0,0.0


,truncated_passages,truncation_rate,truncated_tokens_total,mean_truncated_tokens_all,mean_truncated_tokens_truncated_only
0,5461,0.000212,1080574,0.042045,197.871086


,batch_padded_length_mean,batch_padded_length_variance,batch_padded_length_ci95_low,batch_padded_length_ci95_high,batch_padded_length_min,batch_padded_length_max,actual_visible_tokens_total,padded_tokens_total,padding_overhead_tokens_total,padding_overhead_fraction,effective_token_utilization
0,236.476228,1896.908612,236.263234,236.689222,45,384,2555817045,6077579072,3521762027,0.579468,0.420532


The full scan covers all **25,700,592 TopiOCQA passages**. Mean passage length is 99.49 ANCE tokens and no unknown tokens occur. Only **0.021%** of passages exceed the 384-token limit, making truncation negligible.

At batch size 160, variable passage lengths produce **57.95% padding overhead** and **42.05% effective token utilization**.

### TopiOCQA BM25 index

Inspects and reuses an existing complete Lucene index. Rebuilding is disabled by default and must be explicitly enabled. When requested, the corpus is restored, converted to Pyserini format, indexed with eight threads, and verified again.

In [7]:
REBUILD_TOPIOCQA_BM25_INDEX = False

topiocqa_bm25_status = globals().get("topiocqa_bm25_status_before")
if topiocqa_bm25_status is None:
    print("Inspect TopiOCQA BM25 Lucene index:", TOPIOCQA_BM25_INDEX_DIR, flush=True)
    topiocqa_bm25_status = rocc.inspect_bm25_lucene_index(
        TOPIOCQA_BM25_INDEX_DIR,
        source_passages=rocc.TOPIOCQA_CORPUS_PASSAGES,
        log_file=TOPIOCQA_BM25_LOG_FILE,
    )

display(pd.DataFrame([topiocqa_bm25_status.to_dict()]))

if topiocqa_bm25_status.complete:
    print("TopiOCQA BM25 Lucene index already complete:", TOPIOCQA_BM25_INDEX_DIR)
    topiocqa_bm25_collection = None
    topiocqa_bm25_index = rocc.IndexBuildResult(
        index_dir=TOPIOCQA_BM25_INDEX_DIR,
        command=None,
        built=False,
        log_file=TOPIOCQA_BM25_LOG_FILE,
    )
elif REBUILD_TOPIOCQA_BM25_INDEX:
    if not topiocqa_artifacts.corpus_ready:
        topiocqa_resources = rocc.ensure_topiocqa_resources(
            data_dir=TOPIOCQA_DATA_DIR,
            download_corpus=True,
            progress=True,
            timeout=1000,
        )
        topiocqa_artifacts = rocc.inspect_topiocqa_artifacts(topiocqa_resources)
    topiocqa_bm25_collection = rocc.write_topiocqa_bm25_collection(
        data_dir=TOPIOCQA_DATA_DIR,
        progress=True,
    )
    topiocqa_bm25_index = rocc.build_bm25_lucene_index(
        collection_dir=topiocqa_bm25_collection.output_file.parent,
        index_dir=TOPIOCQA_BM25_INDEX_DIR,
        threads=8,
        progress=True,
        log_file=TOPIOCQA_BM25_LOG_FILE,
        force=topiocqa_bm25_status.exists,
    )
    topiocqa_bm25_status = rocc.inspect_bm25_lucene_index(
        TOPIOCQA_BM25_INDEX_DIR,
        source_passages=topiocqa_bm25_collection.written_passages,
        log_file=TOPIOCQA_BM25_LOG_FILE,
    )
else:
    topiocqa_bm25_collection = None
    topiocqa_bm25_index = None
    print("TopiOCQA BM25 rebuild skipped. Set REBUILD_TOPIOCQA_BM25_INDEX=True to build it.")

if topiocqa_bm25_index is not None:
    display(pd.DataFrame([topiocqa_bm25_index.__dict__]))


,index_dir,exists,loadable,complete,num_docs,source_passages,log_file,log_exit_code,log_indexing_complete,log_indexed_docs,log_empty_docs,error
0,/experiments/data/pyserini_bm25_lucene_topiocqa/lucene_index,True,True,True,25700592,25700592,/experiments/logs/topiocqa_bm25_lucene_index.log,None,False,None,None,None


TopiOCQA BM25 Lucene index already complete: <original-checkout>/experiments/data/pyserini_bm25_lucene_topiocqa/lucene_index


,index_dir,command,built,log_file
0,/experiments/data/pyserini_bm25_lucene_topiocqa/lucene_index,None,False,/experiments/logs/topiocqa_bm25_lucene_index.log


### Inspect the local ANCE index

Place the completed public Drive folder contents directly in the displayed index directory, or build locally below. No cloud credentials are needed.

In [ ]:
topiocqa_artifacts = rocc.inspect_topiocqa_artifacts(topiocqa_resources)
display(pd.DataFrame([topiocqa_artifacts.to_dict()]))
print("Use a fully copied public index at", TOPIOCQA_ANCE_LOCAL_INDEX_DIR,
      "or explicitly start the local build below.")


### Prepare the ANCE collection

If a complete local index is absent, prepare the original text-only passage collection from the public corpus. The following explicit build uses this collection.

In [9]:
topiocqa_ance_index_ready = bool(
    topiocqa_artifacts.ance_index_ready
)
topiocqa_ance_collection_needed = not topiocqa_ance_index_ready

display(pd.DataFrame([{
    "ance_index_ready": topiocqa_ance_index_ready,
    "local_ance_index_ready": topiocqa_artifacts.ance_index_ready,
    "ance_collection_needed": topiocqa_ance_collection_needed,
    "corpus_available": topiocqa_artifacts.corpus_ready,
}]))

if topiocqa_ance_collection_needed:
    if not topiocqa_artifacts.corpus_ready:
        topiocqa_resources = rocc.ensure_topiocqa_resources(
            data_dir=TOPIOCQA_DATA_DIR,
            download_corpus=True,
            progress=True,
            timeout=1000,
        )
        topiocqa_artifacts = rocc.inspect_topiocqa_artifacts(topiocqa_resources)
    topiocqa_ance_collection = rocc.write_topiocqa_ance_collection(
        data_dir=TOPIOCQA_DATA_DIR,
        progress=True,
    )
    display(pd.DataFrame([topiocqa_ance_collection.__dict__]))
else:
    print("TopiOCQA ANCE collection skipped: dense index is already complete locally.")


,ance_index_ready,local_ance_index_ready,azure_ance_index_ready,ance_collection_needed,corpus_available
0,True,False,True,False,True


TopiOCQA ANCE collection skipped: dense index is already complete locally or on Azure.


### Build the local ANCE index

An explicit large job: about 79 GB of float32 vectors plus metadata. Completed matching shards can be resumed. No Azure credentials are needed.

In [ ]:
START_TOPIOCQA_ANCE_BUILD = False
TOPIOCQA_ANCE_DEVICE = "cuda:0" if __import__("torch").cuda.is_available() else "cpu"

if START_TOPIOCQA_ANCE_BUILD and not topiocqa_artifacts.ance_index_ready:
    topiocqa_ance_index = rocc.build_ance_faiss_shards(
        collection_file=topiocqa_ance_collection.output_file,
        output_dir=TOPIOCQA_ANCE_LOCAL_INDEX_DIR,
        batch_size=160, batches_per_shard=5000, max_length=384, dimension=768,
        end_passage=rocc.TOPIOCQA_CORPUS_PASSAGES,
        device=TOPIOCQA_ANCE_DEVICE, progress=True)
    display(pd.DataFrame([topiocqa_ance_index.__dict__]))
else:
    print("Local ANCE build not started. Set START_TOPIOCQA_ANCE_BUILD=True explicitly if required.")


### Completion

The local build above replaces the original external Azure ML execution step. It retains the ANCE encoder, passage order, IDs, 384-token limit, 768 dimensions and float32 inner-product index. Full builds were not executed as part of packaging.